# Week 04 — Python Solution Lab
## Newton's Laws & Free-Body Diagrams

**Companion to `notebooks/Week_04.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_04.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P4` | Static Friction Threshold | friction modelled as a capped function |
| **L2 · Intermediate** | `P5` | Inclined Plane with Friction | critical-angle sweep, mass independence |
| **L3 · Challenge** | `P9` | Three-Block System on an Incline | **linear system** `np.linalg.solve` for FBDs |

---

## L1 · Basic — P4: Static Friction Threshold

> **Problem (Week_04.ipynb, L1 — P4).** A $25.0$ kg box sits on a horizontal surface with
> $\mu_s = 0.40$. What is the minimum horizontal force required to start it moving?

**Diagram → Principle.** FBD: weight down, normal up, applied force horizontal, static friction
opposing. Motion starts when the applied force exceeds $f_{s,\max}$.

**Equation.** $f_{s,\max} = \mu_s N = \mu_s mg$.

**Hand prediction.** $0.40 \times 25.0 \times 9.81 = 98.1$ N.

**What Python adds.** Static friction is *not* a fixed force — it is a **cap**. We write the
friction law as an actual function with the $\min$ built in, apply a ramping force, and plot the
friction response. The kink at $98.1$ N (and the drop to kinetic friction after it) is the whole
concept in one picture.

In [ ]:
# ═══ W04 · L1 · P4 — Static friction is a CAP, not a number ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, mu_s, g = 25.0, 0.40, 9.81
mu_k       = 0.30          # a plausible kinetic value, to show the drop-off
N          = m * g

# --- PREDICT ------------------------------------------------------------
f_max = mu_s * N
print(f"Normal force        N = m*g       = {N:.2f} N")
print(f"Static friction cap f = mu_s * N  = {f_max:.2f} N")
print(f"-> the box starts moving once F exceeds {f_max:.1f} N")

# --- MODEL the friction LAW as a function -------------------------------
def friction(F_applied, moving=False):
    """Returns the friction force that actually acts."""
    if moving:
        return mu_k * N                      # kinetic: fixed magnitude
    return min(F_applied, mu_s * N)          # static: matches F, up to the cap

# --- VERIFY: ramp the applied force and watch the response --------------
F = np.linspace(0, 160, 800)
f_static  = np.minimum(F, f_max)                     # while still at rest
moving    = F > f_max
f_actual  = np.where(moving, mu_k * N, f_static)
a         = np.where(moving, (F - mu_k * N) / m, 0.0)

print("\n  F (N) | friction (N) | moving? | a (m/s^2)")
for Fq in (25, 50, 98, 99, 120, 150):
    mv = Fq > f_max
    fr = friction(Fq, mv)
    print(f"  {Fq:5.0f} | {fr:12.2f} | {str(mv):7s} | {((Fq - fr)/m if mv else 0.0):8.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax1.plot(F, f_static, ls=":", color="grey", label="static cap line")
ax1.plot(F, f_actual, color="#1565c0", lw=2.5, label="friction actually acting")
ax1.axvline(f_max, ls="--", c="crimson", label=f"threshold {f_max:.1f} N")
ax1.set_xlabel("applied force F (N)"); ax1.set_ylabel("friction (N)")
ax1.set_title("friction tracks F, then breaks"); ax1.grid(alpha=.3); ax1.legend(fontsize=8)

ax2.plot(F, a, color="#e65100", lw=2.5)
ax2.axvline(f_max, ls="--", c="crimson")
ax2.set_xlabel("applied force F (N)"); ax2.set_ylabel("a (m/s$^2$)")
ax2.set_title("no motion at all until the cap is exceeded"); ax2.grid(alpha=.3)
plt.suptitle("W04 P4 — 25 kg box, mu_s = 0.40", y=1.03); plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(f_max - 98.1) < 0.1
print(f"[OK] Matches textbook answer: F_min = {f_max:.1f} N")

## L2 · Intermediate — P5: Inclined Plane with Friction

> **Problem (Week_04.ipynb, L2 — P5).** An $8.0$ kg block is placed on a $35^\circ$ incline with
> $\mu_s = 0.45$, $\mu_k = 0.30$. (a) Does it slide? (b) If so, what is its acceleration?
> (c) Released from rest, how fast after sliding $3.0$ m?

**Diagram → Principle.** Rotate axes along the incline. Driving force $mg\sin\theta$ versus the
static cap $\mu_s mg\cos\theta$. Once moving, friction switches to $\mu_k$.

**Equation.** slides if $\tan\theta > \mu_s$; then $a = g(\sin\theta - \mu_k\cos\theta)$.

**Hand prediction.** $\tan 35^\circ = 0.700 > 0.45$, so it slides. $a = 9.81(0.574 - 0.30\cdot0.819) = 3.22$ m/s²; $v = \sqrt{2(3.22)(3.0)} = 4.4$ m/s.

**What Python adds.** The condition $\tan\theta > \mu_s$ means the **critical angle**
$\theta_c = \arctan\mu_s$ is a material property, independent of mass. We compute it, sweep
angle to show where the block starts moving and how $a$ grows, and confirm the mass cancels by
re-running with a $1000\times$ heavier block.

In [ ]:
# ═══ W04 · L2 · P5 — Incline with friction; the critical angle is mass-independent ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, theta_deg, mu_s, mu_k, g = 8.0, 35.0, 0.45, 0.30, 9.81
theta = np.radians(theta_deg)

# --- (a) does it slide? -------------------------------------------------
driving = m * g * np.sin(theta)
cap     = mu_s * m * g * np.cos(theta)
slides  = driving > cap
print(f"(a) driving force  mg sin(theta) = {driving:.2f} N")
print(f"    static cap  mu_s mg cos(th)  = {cap:.2f} N")
print(f"    tan(theta) = {np.tan(theta):.3f}  vs  mu_s = {mu_s:.2f}  ->  "
      f"{'SLIDES' if slides else 'stays put'}")

# --- (b) acceleration once moving (mu_k now) ----------------------------
a = g * (np.sin(theta) - mu_k * np.cos(theta))
print(f"\n(b) a = g(sin th - mu_k cos th) = {a:.3f} m/s^2")

# --- (c) speed after 3.0 m ----------------------------------------------
d = 3.0
v = np.sqrt(2 * a * d)
print(f"(c) v = sqrt(2 a d) = {v:.3f} m/s after {d:.1f} m")

# --- VERIFY: mass really does cancel ------------------------------------
for m_test in (0.008, 8.0, 8000.0):
    a_t = g * (np.sin(theta) - mu_k * np.cos(theta))     # no m anywhere
    slides_t = m_test*g*np.sin(theta) > mu_s*m_test*g*np.cos(theta)
    print(f"    m = {m_test:8.3f} kg -> slides={slides_t}, a = {a_t:.3f} m/s^2")
print("    -> both the threshold and the acceleration are independent of mass.")

# --- Critical angle and an angle sweep ----------------------------------
theta_c = np.degrees(np.arctan(mu_s))
print(f"\nCritical angle theta_c = arctan(mu_s) = {theta_c:.2f} deg")
angs = np.linspace(0, 60, 600)
ar   = np.radians(angs)
a_sweep = np.where(np.tan(ar) > mu_s, g*(np.sin(ar) - mu_k*np.cos(ar)), 0.0)
a_sweep = np.maximum(a_sweep, 0.0)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(angs, a_sweep, color="#1565c0", lw=2.5)
ax.axvline(theta_c, ls="--", c="crimson", label=f"theta_c = {theta_c:.1f} deg")
ax.axvline(theta_deg, ls=":", c="#2e7d32", label=f"this problem: {theta_deg:.0f} deg")
ax.plot(theta_deg, a, "o", color="#2e7d32", ms=9, zorder=5)
ax.set_xlabel("incline angle (deg)"); ax.set_ylabel("a (m/s$^2$)")
ax.set_title("W04 P5 — nothing happens below theta_c, then a rises fast")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert slides
assert abs(a - 3.22) < 0.02, f"a = {a}"
assert abs(v - 4.40) < 0.03, f"v = {v}"
print(f"[OK] Matches textbook answer: slides; a = {a:.2f} m/s^2; v = {v:.2f} m/s")

## L3 · Challenge — P9: Three-Block System on an Incline

> **Problem (Week_04.ipynb, L3 — P9).** Three blocks ($m_1 = 2.0$, $m_2 = 3.0$, $m_3 = 5.0$ kg)
> are connected by strings on a frictionless $30^\circ$ incline, $m_3$ at the top, $m_2$ in the
> middle, $m_1$ at the bottom. Released from rest. Find (a) the acceleration, (b) $T_{12}$,
> (c) $T_{23}$.

**Diagram → Principle.** One FBD per block, all sharing a single acceleration $a$ down the
incline. That is **three equations in three unknowns** ($a$, $T_{12}$, $T_{23}$) — the classic
connected-body setup.

**Equation.** Per block, taking down-incline as positive:
$m_ig\sin\theta - f_i + T_{\rm below} - T_{\rm above} = m_ia$.
Summing the three eliminates the tensions and gives $a = g\sin\theta$ for the frictionless case.

**Hand prediction.** $a = 9.81\sin30^\circ = 4.905$ m/s², and both tensions come out **zero**:
each block, left alone on a frictionless incline, would already accelerate at exactly
$g\sin\theta$, so no string has to pull on anything.

**What Python adds.** Rather than trusting the shortcut, we assemble the FBD equations as a
genuine **linear system** $A\vec u = \vec b$ and let `numpy.linalg.solve` find $a$, $T_{12}$,
$T_{23}$ together. That is the method that keeps working when friction differs per block — which
we then demonstrate, showing the tensions come alive the moment the blocks are no longer
identical in their friction.

In [ ]:
# ═══ W04 · L3 · P9 — Connected blocks as a linear system A u = b ═══
import numpy as np

g, theta = 9.81, np.radians(30.0)
m1, m2, m3 = 2.0, 3.0, 5.0

def solve_chain(masses, mus, g=9.81, theta=theta):
    """
    Blocks numbered 1 (bottom) .. n (top), connected by n-1 strings.
    Positive axis = DOWN the incline. Unknowns u = [a, T_12, T_23, ...].
    Block i:  m_i g sin(th) - f_i + (T_below - T_above ... ) = m_i a
      - bottom block  : pulled back by T_12 (the block above holds it? no - it TRAILS)
    Sign convention used here, taking tension as tending to hold each block back
    from its neighbour above and pull it forward from its neighbour below.
    """
    n = len(masses)
    A = np.zeros((n, n)); b = np.zeros(n)
    for i, (m, mu) in enumerate(zip(masses, mus)):
        A[i, 0] = m                                   # coefficient of a
        if i > 0:     A[i, i]     = -1.0               # string to the block below
        if i < n - 1: A[i, i + 1] = +1.0               # string to the block above
        b[i] = m * g * np.sin(theta) - mu * m * g * np.cos(theta)
    return np.linalg.solve(A, b)

# --- (a)-(c) frictionless case ------------------------------------------
u = solve_chain([m1, m2, m3], [0.0, 0.0, 0.0])
a, T12, T23 = u
print("FRICTIONLESS (the stated problem)")
print(f"(a) a   = {a:.3f} m/s^2      [check: g sin(30) = {g*np.sin(theta):.3f}]")
print(f"(b) T12 = {T12:.3f} N")
print(f"(c) T23 = {T23:.3f} N")
print("    -> both tensions are ZERO. Note what that does and does not mean: the strings")
print("       are UNLOADED, not necessarily geometrically slack. They can stay perfectly")
print("       taut and simply carry no force, because every block would already accelerate")
print("       at g sin(theta) on its own -- there is no constraint left for them to enforce.")

# --- VERIFY the linear-system solution against the closed form ----------
assert abs(a - g * np.sin(theta)) < 1e-12
assert abs(T12) < 1e-12 and abs(T23) < 1e-12

# --- WHY BOTHER with the matrix? Add unequal friction -------------------
print("\nSAME CODE, now with different friction under each block")
mus = [0.10, 0.25, 0.40]        # e.g. three different materials
a2, T12b, T23b = solve_chain([m1, m2, m3], mus)
print(f"    mu = {mus}")
print(f"    a   = {a2:.3f} m/s^2")
print(f"    T12 = {T12b:.3f} N     T23 = {T23b:.3f} N   <- now genuinely loaded")

# system-level cross-check: total driving force / total mass
M = m1 + m2 + m3
a_sys = (M*g*np.sin(theta) - sum(mu*m*g*np.cos(theta) for mu, m in zip(mus, [m1,m2,m3]))) / M
print(f"    system check: (sum F)/(sum m) = {a_sys:.3f} m/s^2  -> agrees")
assert abs(a2 - a_sys) < 1e-9

# per-block Newton check on the friction case
for i, (m, mu, T_b, T_a) in enumerate(
        zip([m1,m2,m3], mus, [0.0, T12b, T23b], [T12b, T23b, 0.0]), start=1):
    net = m*g*np.sin(theta) - mu*m*g*np.cos(theta) + T_b - T_a
    assert abs(net - m*a2) < 1e-9, f"block {i} FBD inconsistent"
print("    every individual FBD balances to m_i * a. [verified]")

print(f"\n[OK] Matches textbook answer: a = {a:.2f} m/s^2, T12 = T23 = 0 N "
      "(frictionless => unloaded strings).")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_04.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
